## 1. Imports

In [ ]:
import random
from pathlib import Path
import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from sklearn.svm import LinearSVC
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

random.seed(42); np.random.seed(42)
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['image.cmap'] = 'gray'

## 2. Configuració

In [ ]:
RAW_DIR = Path('data/raw')
VALID_EXTS = {'.jpg', '.jpeg', '.png', '.bmp'}

# HOG window: aspect ratio ~3:1 like a plate
WIN_W, WIN_H = 96, 32
WIN_SIZE = (WIN_W, WIN_H)

HOG = cv2.HOGDescriptor(
    _winSize=(WIN_W, WIN_H),
    _blockSize=(16, 16),
    _blockStride=(8, 8),
    _cellSize=(8, 8),
    _nbins=9,
)

# Sliding-window detection
STEP = 8
SCALES = [0.5, 0.75, 1.0, 1.25, 1.5, 2.0]
SCORE_THRESHOLD = 0.5
NMS_IOU = 0.3

# Negative samples per image (random crops with IoU<0.1 vs GT)
N_NEG_PER_IMAGE = 15

print(f'HOG descriptor size: {HOG.getDescriptorSize()}')

## 3. Parser d'annotations

Format: `name <TAB> x <TAB> y <TAB> w <TAB> h <TAB> plate_text`. Acceptem tant TAB com espais com a separador per robustesa.

In [ ]:
def parse_annotation(txt_path):
    # Returns a list of (x_min, y_min, x_max, y_max) in pixels.
    boxes = []
    with open(txt_path) as f:
        for line in f:
            line = line.strip()
            if not line: continue
            # try tab first, fallback to whitespace
            parts = line.split('\t')
            if len(parts) < 5:
                parts = line.split()
            if len(parts) < 5:
                continue
            try:
                x = int(parts[1]); y = int(parts[2])
                w = int(parts[3]); h = int(parts[4])
            except ValueError:
                continue
            if w > 0 and h > 0:
                boxes.append((x, y, x+w, y+h))
    return boxes

def get_txt_path(img_path):
    return img_path.with_suffix('.txt')

## 4. Sanity check — verifiquem que les annotations es llegeixen bé

In [ ]:
all_images = sorted([p for p in RAW_DIR.iterdir() if p.suffix.lower() in VALID_EXTS])
print(f'Found {len(all_images)} images in {RAW_DIR}')

# Count how many have .txt
with_txt = [p for p in all_images if get_txt_path(p).exists()]
print(f'  with .txt annotation: {len(with_txt)}')
print(f'  without .txt:         {len(all_images) - len(with_txt)}')

# Show 3 random examples with their GT boxes overlaid
sample = random.sample(with_txt, min(3, len(with_txt)))
fig, axes = plt.subplots(1, len(sample), figsize=(15, 5))
if len(sample) == 1: axes = [axes]

for ax, img_path in zip(axes, sample):
    img = cv2.imread(str(img_path))
    boxes = parse_annotation(get_txt_path(img_path))
    ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    for (x1, y1, x2, y2) in boxes:
        ax.add_patch(patches.Rectangle((x1, y1), x2-x1, y2-y1,
                     linewidth=2, edgecolor='red', facecolor='none'))
    ax.set_title(f'{img_path.name} -- {len(boxes)} GT box(es)')
    ax.axis('off')

plt.tight_layout(); plt.show()

## 5. Generar positius i negatius

**Positius**: retall de cada GT box (96x32) + flip horitzontal (data augmentation).
**Negatius**: retalls aleatoris que no solapen amb cap GT (IoU < 0.1).

In [ ]:
def iou(a, b):
    x1 = max(a[0], b[0]); y1 = max(a[1], b[1])
    x2 = min(a[2], b[2]); y2 = min(a[3], b[3])
    inter = max(0, x2-x1) * max(0, y2-y1)
    aa = (a[2]-a[0]) * (a[3]-a[1])
    bb = (b[2]-b[0]) * (b[3]-b[1])
    return inter / (aa + bb - inter + 1e-9)

def compute_hog(img):
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY) if img.ndim == 3 else img
    if gray.shape != (WIN_H, WIN_W):
        gray = cv2.resize(gray, WIN_SIZE)
    return HOG.compute(gray).flatten()

positives, negatives = [], []

for img_path in with_txt:
    img = cv2.imread(str(img_path))
    if img is None: continue
    H, W = img.shape[:2]
    gt_boxes = parse_annotation(get_txt_path(img_path))
    if not gt_boxes: continue

    # positives
    for (x1, y1, x2, y2) in gt_boxes:
        x1, y1 = max(0, x1), max(0, y1)
        x2, y2 = min(W, x2), min(H, y2)
        crop = img[y1:y2, x1:x2]
        if crop.size == 0: continue
        positives.append(compute_hog(crop))
        positives.append(compute_hog(cv2.flip(crop, 1)))

    # negatives
    added, tries = 0, 0
    while added < N_NEG_PER_IMAGE and tries < 200:
        tries += 1
        scale = random.uniform(0.7, 1.5)
        w = int(WIN_W * scale); h = int(WIN_H * scale)
        if w >= W or h >= H: continue
        x = random.randint(0, W - w); y = random.randint(0, H - h)
        cand = (x, y, x+w, y+h)
        if any(iou(cand, gt) > 0.1 for gt in gt_boxes): continue
        crop = img[y:y+h, x:x+w]
        negatives.append(compute_hog(crop))
        added += 1

print(f'Positives: {len(positives)} (with horizontal flip aug)')
print(f'Negatives: {len(negatives)}')

X = np.array(positives + negatives, dtype=np.float32)
y = np.array([1]*len(positives) + [0]*len(negatives), dtype=np.int32)
print(f'X: {X.shape}, y: {y.shape}')

## 6. Entrenar SVM lineal

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

clf = Pipeline([
    ('scaler', StandardScaler()),
    ('svm', LinearSVC(C=1.0, max_iter=5000, dual='auto')),
])

clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)
print(classification_report(y_test, y_pred, target_names=['no-plate', 'plate']))

## 7. Detecció: finestra lliscant + piràmide multi-escala

In [ ]:
def sliding_window_detect(img_bgr, clf, threshold=SCORE_THRESHOLD):
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    detections = []
    for scale in SCALES:
        new_w = int(gray.shape[1] / scale)
        new_h = int(gray.shape[0] / scale)
        if new_w < WIN_W or new_h < WIN_H: continue
        resized = cv2.resize(gray, (new_w, new_h))

        coords, feats = [], []
        for yy in range(0, new_h - WIN_H + 1, STEP):
            for xx in range(0, new_w - WIN_W + 1, STEP):
                feats.append(compute_hog(resized[yy:yy+WIN_H, xx:xx+WIN_W]))
                coords.append((xx, yy))
        if not feats: continue
        feats = np.array(feats, dtype=np.float32)
        scores = clf.decision_function(feats)
        for (xx, yy), s in zip(coords, scores):
            if s > threshold:
                detections.append((int(xx*scale), int(yy*scale),
                                   int(WIN_W*scale), int(WIN_H*scale), float(s)))
    return detections

## 8. Non-Maximum Suppression

In [ ]:
def nms(detections, iou_thresh=NMS_IOU):
    if not detections: return []
    arr = np.array([(x, y, x+w, y+h, s) for (x,y,w,h,s) in detections], dtype=np.float32)
    x1, y1, x2, y2, scores = arr.T
    areas = (x2 - x1) * (y2 - y1)
    order = scores.argsort()[::-1]
    keep = []
    while order.size > 0:
        i = order[0]; keep.append(i)
        xx1 = np.maximum(x1[i], x1[order[1:]])
        yy1 = np.maximum(y1[i], y1[order[1:]])
        xx2 = np.minimum(x2[i], x2[order[1:]])
        yy2 = np.minimum(y2[i], y2[order[1:]])
        inter = np.maximum(0, xx2-xx1) * np.maximum(0, yy2-yy1)
        iou_ = inter / (areas[i] + areas[order[1:]] - inter + 1e-9)
        order = order[1:][iou_ < iou_thresh]
    out = []
    for i in keep:
        a = arr[i]
        out.append((int(a[0]), int(a[1]), int(a[2]-a[0]), int(a[3]-a[1]), float(a[4])))
    return out

## 9. Detecció sobre 10 imatges aleatòries

In [ ]:
n_to_pick = min(50, len(all_images))
selected_paths = random.sample(all_images, n_to_pick)

results = []
for path in selected_paths:
    img = cv2.imread(str(path))
    if img is None: continue
    raw = sliding_window_detect(img, clf)
    final = nms(raw)
    results.append((path, img, raw, final))
    print(f'{path.name:30s} -> {len(raw):4d} raw -> {len(final):2d} after NMS')

## 10. Visualització detallada

Vermell = GT (ground truth), Verd = deteccions del SVM.

In [ ]:
def show_detection(img, raw, final, gt, title=''):
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    fig.suptitle(title, fontsize=14, fontweight='bold')

    axes[0].imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    for (x, y, w, h, s) in raw:
        axes[0].add_patch(patches.Rectangle((x, y), w, h, linewidth=1,
                          edgecolor='yellow', facecolor='none', alpha=0.4))
    for (x1, y1, x2, y2) in gt:
        axes[0].add_patch(patches.Rectangle((x1, y1), x2-x1, y2-y1,
                          linewidth=2, edgecolor='red', facecolor='none'))
    axes[0].set_title(f'Raw ({len(raw)}) + GT (red)')
    axes[0].axis('off')

    axes[1].imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    for (x, y, w, h, s) in final:
        axes[1].add_patch(patches.Rectangle((x, y), w, h, linewidth=2,
                          edgecolor='lime', facecolor='none'))
        axes[1].text(x, y-3, f'{s:.2f}', color='lime', fontsize=9, fontweight='bold')
    for (x1, y1, x2, y2) in gt:
        axes[1].add_patch(patches.Rectangle((x1, y1), x2-x1, y2-y1,
                          linewidth=2, edgecolor='red', facecolor='none', linestyle='--'))
    axes[1].set_title(f'After NMS ({len(final)}) + GT (red dashed)')
    axes[1].axis('off')

    plt.tight_layout(); plt.show()

for path, img, raw, final in results:
    txt = get_txt_path(path)
    gt = parse_annotation(txt) if txt.exists() else []
    show_detection(img, raw, final, gt, title=path.name)

## 11. Vista resum

In [ ]:
n = len(results); cols = 2; rows = (n + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(14, 5*rows))
axes = np.atleast_2d(axes).flatten()
for ax, (path, img, raw, final) in zip(axes, results):
    ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    for (x, y, w, h, s) in final:
        ax.add_patch(patches.Rectangle((x, y), w, h, linewidth=2, edgecolor='lime', facecolor='none'))
    ax.set_title(f'{path.name} -- {len(final)} boxes')
    ax.axis('off')
for ax in axes[len(results):]: ax.axis('off')
plt.tight_layout(); plt.show()

## 12. Notes per ajustar

- **Massa false positives**: puja `SCORE_THRESHOLD` a `0.5` o `1.0`.
- **No detecta res**: baixa a `-0.5`. Si tampoc, mira el `classification_report` -- potser pocs positius o desbalanceig massa gran.
- **Massa lent**: redueix `SCALES` (per exemple només `[0.75, 1.0, 1.5]`) o augmenta `STEP` a 16.
- **Millorar precisió**: afegir hard-negative mining (entrenar, mirar FPs en imatges raw, afegir-los com a negatius, reentrenar).